In [0]:
from pyspark.sql import SparkSession
spark=SparkSession.builder.appName('DateDimension').getOrCreate()

In [0]:
# generate date range
from pyspark.sql.functions import explode, sequence, to_date, lit

start_date="2023-01-01"
end_date="2025-12-31"

date_df=spark.sql(f"""
                  SELECT explode(sequence(to_date('{start_date}'), to_date('{end_date}'), interval 1 day)) as date
                  """)

In [0]:
# add dae attributes

from pyspark.sql.functions import (year, month, dayofmonth, dayofweek, weekofyear, date_format, quarter, when)

dim_date=date_df.select("date",
                        year("date").alias("year"),
                        month("date").alias("month"),
                        dayofmonth("date").alias("day"),
                        dayofweek("date").alias("day_of_week"),
                        weekofyear("date").alias("week_of_year"),
                        quarter("date").alias("quarter"),
                        date_format("date", "EEE").alias("day-name"),
                        date_format("date", "MMM").alias("month_name"),
                        when(dayofweek("date").isin([1,7]),True).otherwise(False).alias("is_weekend")
                        )

##### surrogate key - an artificial (system generated) unique identifier foreach row in a table
problem with natural keys - values can change in some domains, can be complex with ,ultiple columns, slower joins especially strings
surrogate keys are used for faster joins, consistent across systems, simplifies data modeling, required in star schema 
example date_key = format -> YYYYMMDD, date format -> YYYY-MM-DD, to convert to date_key the value of 10000 is used

In [0]:
# adding surrogate key
from pyspark.sql.functions import monotonically_increasing_id

dim_date=dim_date.withColumn("date_key",(year("date")*10000+month("date")))

In [0]:
# reordering columns
dim_date =dim_date.select("date_key", "date", "year", "quarter", "month", 
                          "month_name", "week_of_year","day","day-name","day_of_week","is_weekend"
                          )
dim_date.show(3)